# Lab 2: Predicción de coeficientes de presión superficial

**Objetivo:** desarrollar y evaluar modelos de regresión que permitan estimar los coeficientes de presión superficial `Cp_mean`, `Cp_rms`, `Cp_min` y `Cp_max` a partir de variables geométricas, aerodinámicas y espaciales obtenidas de ensayos experimentales realizados en los túneles de viento de la Tokyo Polytechnic University, TPU.

También se analizará la importancia de las variables de entrada para identificar qué factores dominan el comportamiento de las presiones superficiales sobre las fachadas y cubiertas.

## Recorrido del notebook

1. Contexto físico y descripción del conjunto de datos.
2. Configuración del entorno.
3. Carga del conjunto de datos consolidado.
4. Control de calidad de los datos.
5. Estadísticas descriptivas.
6. Distribución de las variables objetivo.
7. Correlación entre variables.
8. Relación física entre ángulo del viento y presión.
9. Distribución espacial de las presiones.
10. Partición agrupada de entrenamiento y prueba.
11. Modelo de referencia.
12. Entrenamiento de Random Forest.
13. Evaluación mediante MAE, RMSE y R².
14. Importancia de variables por permutación.
15. Validación visual.
16. Análisis de estabilidad.
17. Reflexión y conclusiones.

## Contexto del conjunto de datos TPU

La Tokyo Polytechnic University dispone de bases de datos aerodinámicas obtenidas mediante ensayos experimentales en túneles de viento. Estas bases contienen mediciones de presión sobre superficies de modelos de edificios con diferentes geometrías y direcciones de incidencia del viento.

| Variable | Unidad o tipo | Interpretación física |
|---|---:|---|
| `H` | longitud | Altura del edificio |
| `B` | longitud | Ancho del edificio |
| `D` | longitud | Profundidad del edificio |
| `H_B` | adimensional | Relación de aspecto `H/B` |
| `theta` | grados | Dirección de incidencia del viento |
| `X` | longitud o coordenada normalizada | Posición horizontal del sensor |
| `Y` | longitud o coordenada normalizada | Posición transversal del sensor |
| `Z` | longitud o coordenada normalizada | Altura del sensor |
| `geometry_id` | categórica | Identificador de la geometría o modelo |
| `sensor_id` | categórica | Identificador de la toma de presión |
| `cp_mean` | adimensional | Coeficiente medio de presión |
| `cp_rms` | adimensional | Intensidad de las fluctuaciones de presión |
| `cp_min` | adimensional | Pico mínimo o succión crítica observada |
| `cp_max` | adimensional | Pico máximo o sobrepresión observada |

Los coeficientes de presión son variables continuas. Por tanto, el problema se formula como una tarea de **regresión supervisada**.

Se entrenarán modelos independientes para cada variable objetivo. Esta separación permite analizar de manera específica la presión media, la variabilidad y los extremos de presión.

import sys
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold,
    cross_val_score,
)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("ignore")

%matplotlib inline

sns.set_theme(
    style="whitegrid",
    context="notebook",
)

RANDOM_STATE = 42

print("✅ Entorno preparado.")
print(f"Python: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")
print(f"Semilla aleatoria: {RANDOM_STATE}")

## 1. Contexto aerodinámico y Machine Learning

La predicción de coeficientes de presión superficial es un problema de regresión porque las variables objetivo son valores continuos.

En ingeniería de viento, los diferentes coeficientes representan fenómenos distintos:

- `cp_mean` describe la distribución media de la presión.
- `cp_rms` representa la intensidad de las fluctuaciones.
- `cp_min` permite estudiar las succiones locales críticas.
- `cp_max` representa los máximos de sobrepresión observados.

Los valores negativos de Cp representan succión y los valores positivos representan presión hacia la superficie.

La predicción de `cp_min` es especialmente importante en zonas próximas a bordes, esquinas y cubiertas, donde la separación del flujo y los vórtices pueden producir succiones locales intensas.

Los valores `cp_min` y `cp_max` utilizados en este estudio corresponden a los extremos observados durante la duración de cada ensayo. No deben interpretarse automáticamente como valores extremos de diseño asociados con un periodo específico de retorno.

TIPO_PROBLEMA = "regresion"

VARIABLES_OBJETIVO = [
    "cp_mean",
    "cp_rms",
    "cp_min",
    "cp_max",
]

VARIABLES_ENTRADA = [
    "H",
    "B",
    "D",
    "H_B",
    "theta",
    "X",
    "Y",
    "Z",
]

COLUMNA_GRUPO = "geometry_id"

print(f"Tipo de problema: {TIPO_PROBLEMA}")
print(f"Variables de entrada: {VARIABLES_ENTRADA}")
print(f"Variables objetivo: {VARIABLES_OBJETIVO}")
print(f"Variable de agrupación: {COLUMNA_GRUPO}")

## 2. Carga del conjunto de datos

El notebook utiliza la matriz consolidada generada a partir de los archivos experimentales de la TPU.

La ruta principal esperada es:

`data/processed.parquet`

El formato Parquet es preferible a CSV porque conserva los tipos de datos, ocupa menos espacio y permite una lectura más eficiente.

Si no existe el archivo Parquet, el notebook intentará cargar:

`data/processed.csv`

RUTA_PARQUET = Path("data/processed.parquet")
RUTA_CSV = Path("data/processed.csv")

if RUTA_PARQUET.exists():
    df = pd.read_parquet(RUTA_PARQUET)
    RUTA_DATOS = RUTA_PARQUET

elif RUTA_CSV.exists():
    df = pd.read_csv(RUTA_CSV)
    RUTA_DATOS = RUTA_CSV

else:
    raise FileNotFoundError(
        "No se encontró data/processed.parquet ni "
        "data/processed.csv.\n"
        "Ejecute primero el módulo build_dataset o coloque "
        "el archivo consolidado dentro de data/."
    )

print(f"Archivo cargado: {RUTA_DATOS}")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

N_FILAS_HEAD = 5

print(f"Primeras {N_FILAS_HEAD} observaciones:")

display(df.head(N_FILAS_HEAD))

COLUMNAS_REQUERIDAS = (
    VARIABLES_ENTRADA
    + VARIABLES_OBJETIVO
    + [COLUMNA_GRUPO]
)

columnas_faltantes = [
    columna
    for columna in COLUMNAS_REQUERIDAS
    if columna not in df.columns
]

if columnas_faltantes:
    raise KeyError(
        "Faltan columnas obligatorias en el conjunto de datos: "
        f"{columnas_faltantes}"
    )

print("✅ Todas las columnas requeridas están disponibles.")

## 3. Calidad de los datos

Antes del entrenamiento se deben revisar:

- Valores faltantes.
- Duplicados.
- Valores infinitos.
- Rangos geométricos.
- Direcciones del viento.
- Coordenadas de sensores.
- Valores extremos de Cp.
- Cantidad de geometrías y ensayos.

Un valor extremo de presión no debe eliminarse automáticamente. En ingeniería de viento, una succión grande puede corresponder a un fenómeno físico real localizado en un borde o una esquina.

Los valores extremos deben contrastarse con:

1. La posición del sensor.
2. La dirección del viento.
3. La geometría del edificio.
4. Los sensores cercanos.
5. La calidad experimental del ensayo.

print("Tipos de datos:")
display(df[COLUMNAS_REQUERIDAS].dtypes.to_frame("tipo"))

calidad = pd.DataFrame({
    "tipo": df[COLUMNAS_REQUERIDAS].dtypes.astype(str),
    "nulos": df[COLUMNAS_REQUERIDAS].isna().sum(),
    "nulos_porcentaje": (
        df[COLUMNAS_REQUERIDAS].isna().mean() * 100
    ),
    "valores_unicos": df[COLUMNAS_REQUERIDAS].nunique(),
})

display(calidad.sort_values(
    "nulos_porcentaje",
    ascending=False,
))

porcentaje_nulos = (
    df[COLUMNAS_REQUERIDAS]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(11, 5))

sns.barplot(
    x=porcentaje_nulos.index,
    y=porcentaje_nulos.values,
    color="#3498db",
    ax=ax,
)

ax.set_xlabel("Variable")
ax.set_ylabel("Datos faltantes (%)")
ax.set_title("Porcentaje de datos faltantes")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

n_duplicados = int(df.duplicated().sum())

columnas_numericas = (
    VARIABLES_ENTRADA
    + VARIABLES_OBJETIVO
)

n_infinitos = int(
    np.isinf(
        df[columnas_numericas]
        .select_dtypes(include=np.number)
        .to_numpy()
    ).sum()
)

print(f"Filas duplicadas: {n_duplicados:,}")
print(f"Valores infinitos: {n_infinitos:,}")

if n_duplicados > 0:
    print(
        "⚠️ Existen filas duplicadas. Debe verificarse si "
        "representan repeticiones experimentales válidas."
    )

COLUMNA_REVISAR = "theta"

print(
    f"Resumen de la variable crítica: {COLUMNA_REVISAR}"
)

display(df[COLUMNA_REVISAR].describe())

## 4. Estadísticas descriptivas

Las estadísticas descriptivas permiten conocer el dominio cubierto por los ensayos.

Esto es importante porque un modelo de Random Forest puede interpolar dentro del rango observado, pero no es apropiado para extrapolar hacia:

- Geometrías mucho más altas o anchas.
- Relaciones `H/B` no observadas.
- Direcciones del viento ausentes.
- Coordenadas fuera de las superficies instrumentadas.
- Valores extremos de presión de otros regímenes de flujo.

COLUMNAS_RESUMEN = (
    VARIABLES_ENTRADA
    + VARIABLES_OBJETIVO
)

resumen = df[COLUMNAS_RESUMEN].describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
    ]
).T

display(resumen)

n_geometrias = df[COLUMNA_GRUPO].nunique()

n_sensores = (
    df["sensor_id"].nunique()
    if "sensor_id" in df.columns
    else np.nan
)

n_angulos = df["theta"].nunique()

print(f"Geometrías: {n_geometrias:,}")
print(f"Sensores diferentes: {n_sensores}")
print(f"Direcciones de viento: {n_angulos:,}")
print(f"Observaciones totales: {len(df):,}")

conteo_geometrias = (
    df[COLUMNA_GRUPO]
    .astype(str)
    .value_counts()
    .sort_values(ascending=False)
)

display(
    conteo_geometrias
    .rename("n_observaciones")
    .head(20)
    .to_frame()
)

fig, ax = plt.subplots(figsize=(10, 7))

conteo_geometrias.head(25).sort_values().plot(
    kind="barh",
    ax=ax,
    color="#2980b9",
)

ax.set_xlabel("Número de observaciones")
ax.set_ylabel("Geometría")
ax.set_title(
    "Cantidad de observaciones por geometría\n"
    "25 geometrías con mayor representación"
)

plt.tight_layout()
plt.show()

## 5. Distribución de las variables objetivo

Las distribuciones de los coeficientes permiten detectar:

- Asimetría.
- Colas extremas.
- Succiones críticas.
- Diferencias de escala.
- Posibles errores de medición.
- Concentración de observaciones en determinados rangos.

Para `cp_min` se espera una cola negativa debido a las succiones producidas por separación del flujo y desprendimiento de vórtices.

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 9),
)

axes = axes.ravel()

colores = {
    "cp_mean": "#3498db",
    "cp_rms": "#f39c12",
    "cp_min": "#e74c3c",
    "cp_max": "#2ecc71",
}

for ax, target in zip(axes, VARIABLES_OBJETIVO):

    sns.histplot(
        data=df,
        x=target,
        bins=40,
        kde=True,
        color=colores[target],
        ax=ax,
    )

    ax.set_title(f"Distribución de {target}")
    ax.set_xlabel(target)
    ax.set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

CUANTILES = [
    0.01,
    0.05,
    0.50,
    0.95,
    0.99,
]

cuantiles_objetivos = (
    df[VARIABLES_OBJETIVO]
    .quantile(CUANTILES)
    .T
)

cuantiles_objetivos.columns = [
    "P01",
    "P05",
    "P50",
    "P95",
    "P99",
]

display(cuantiles_objetivos)

UMBRAL_SUCCION = -1.0

n_succiones_criticas = int(
    (df["cp_min"] <= UMBRAL_SUCCION).sum()
)

porcentaje_succiones = (
    n_succiones_criticas
    / len(df)
    * 100
)

print(
    f"Observaciones con cp_min ≤ {UMBRAL_SUCCION}: "
    f"{n_succiones_criticas:,}"
)

print(
    f"Porcentaje del conjunto: "
    f"{porcentaje_succiones:.2f}%"
)

## 6. Correlación entre variables

Se utiliza la correlación de Spearman porque permite detectar relaciones monótonas y es menos sensible a valores extremos que la correlación de Pearson.

Una correlación pequeña no demuestra ausencia de influencia física. La distribución de presiones puede depender de interacciones no lineales entre:

- Geometría.
- Ángulo del viento.
- Fachada.
- Altura del sensor.
- Distancia a bordes y esquinas.

TOP_N_CORR = 5

for target in VARIABLES_OBJETIVO:

    correlaciones_target = (
        corr[target]
        .drop(VARIABLES_OBJETIVO)
        .sort_values(
            key=np.abs,
            ascending=False,
        )
        .head(TOP_N_CORR)
    )

    print(f"\nTop correlaciones con {target}:")

    for variable, valor in correlaciones_target.items():
        print(f"  {variable}: ρ = {valor:.3f}")

## 7. Relación física entre el ángulo del viento y Cp

Pequeños cambios en la dirección del viento pueden modificar:

- El punto de estancamiento.
- La fachada de barlovento.
- La posición de las zonas de separación.
- La intensidad de las succiones.
- La ubicación de los vórtices de esquina.

Por este motivo, se estudia la variación de los objetivos con el ángulo `theta`.

estadisticas_angulo = (
    df.groupby("theta")[VARIABLES_OBJETIVO]
    .agg(["mean", "std"])
    .sort_index()
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(14, 9),
    sharex=True,
)

axes = axes.ravel()

for ax, target in zip(
    axes,
    VARIABLES_OBJETIVO,
):

    x = estadisticas_angulo.index.to_numpy(
        dtype=float
    )

    media = estadisticas_angulo[
        (target, "mean")
    ].to_numpy(dtype=float)

    desviacion = (
        estadisticas_angulo[(target, "std")]
        .fillna(0)
        .to_numpy(dtype=float)
    )

    ax.plot(
        x,
        media,
        marker="o",
        markersize=3,
        color=colores[target],
        label="Media",
    )

    ax.fill_between(
        x,
        media - desviacion,
        media + desviacion,
        color=colores[target],
        alpha=0.20,
        label="±1 desviación estándar",
    )

    ax.set_title(f"{target} según el ángulo")
    ax.set_xlabel("Dirección del viento θ (°)")
    ax.set_ylabel(target)
    ax.legend()

plt.tight_layout()
plt.show()

## 8. Distribución espacial de las presiones

Las coordenadas de las tomas de presión permiten estudiar la distribución de Cp sobre la superficie.

Las succiones más importantes suelen localizarse cerca de:

- Bordes de ataque.
- Esquinas.
- Cambios de superficie.
- Zonas altas.
- Cubiertas.
- Regiones de separación del flujo.

El gráfico `X-Z` es una representación exploratoria. Si el conjunto contiene diferentes fachadas, debe utilizarse `Y` o una identificación de superficie para evitar mezclar sensores que pertenecen a caras distintas.

COLUMNAS_MAPA = [
    "X",
    "Z",
    "cp_min",
]

datos_mapa = (
    df[COLUMNAS_MAPA]
    .dropna()
    .copy()
)

MAX_PUNTOS_MAPA = 30000

if len(datos_mapa) > MAX_PUNTOS_MAPA:
    datos_mapa = datos_mapa.sample(
        MAX_PUNTOS_MAPA,
        random_state=RANDOM_STATE,
    )

fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(
    datos_mapa["X"],
    datos_mapa["Z"],
    c=datos_mapa["cp_min"],
    cmap="coolwarm",
    s=12,
    alpha=0.70,
)

ax.set_xlabel("Coordenada X")
ax.set_ylabel("Coordenada Z")
ax.set_title(
    "Distribución espacial del Cp mínimo"
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Cp mínimo")

plt.tight_layout()
plt.show()

## 9. Partición agrupada de entrenamiento y prueba

No se utiliza una división aleatoria convencional de filas.

Las tomas de presión de una misma geometría comparten:

- Condiciones experimentales.
- Dimensiones.
- Relaciones espaciales.
- Patrones aerodinámicos.
- Direcciones del viento.

Si sensores de una misma geometría aparecen simultáneamente en entrenamiento y prueba, el modelo puede producir resultados excesivamente optimistas.

Para evitar esta fuga de información se utiliza `GroupShuffleSplit`, reservando geometrías completas para la evaluación final.

TARGET = "cp_mean"

columnas_modelo = (
    VARIABLES_ENTRADA
    + [TARGET, COLUMNA_GRUPO]
)

df_modelo = (
    df[columnas_modelo]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(
        subset=[TARGET, COLUMNA_GRUPO]
    )
    .copy()
)

X = df_modelo[VARIABLES_ENTRADA]
y = df_modelo[TARGET]
grupos = df_modelo[COLUMNA_GRUPO].astype(str)

print(f"Objetivo seleccionado: {TARGET}")
print(f"X: {X.shape}")
print(f"y: {y.shape}")
print(
    f"Grupos disponibles: "
    f"{grupos.nunique()}"
)

TEST_SIZE = 0.20

group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

indice_train, indice_test = next(
    group_split.split(
        X,
        y,
        groups=grupos,
    )
)

X_train = X.iloc[indice_train].copy()
X_test = X.iloc[indice_test].copy()

y_train = y.iloc[indice_train].copy()
y_test = y.iloc[indice_test].copy()

grupos_train = grupos.iloc[indice_train]
grupos_test = grupos.iloc[indice_test]

print(f"Filas de entrenamiento: {len(X_train):,}")
print(f"Filas de prueba: {len(X_test):,}")

print(
    f"Geometrías de entrenamiento: "
    f"{grupos_train.nunique()}"
)

print(
    f"Geometrías de prueba: "
    f"{grupos_test.nunique()}"
)

grupos_compartidos = (
    set(grupos_train.unique())
    .intersection(
        set(grupos_test.unique())
    )
)

print(
    f"Geometrías presentes en ambos conjuntos: "
    f"{len(grupos_compartidos)}"
)

assert len(grupos_compartidos) == 0, (
    "Existe fuga de información entre "
    "entrenamiento y prueba."
)

print("✅ No existen geometrías compartidas.")

## 10. Modelo de referencia

Antes de evaluar Random Forest se entrena un modelo base que siempre predice la mediana del conjunto de entrenamiento.

El modelo de Machine Learning debe superar claramente esta referencia. De lo contrario, no existe evidencia suficiente de que haya aprendido una relación útil entre las variables de entrada y Cp.

def calcular_metricas(y_real, y_estimado):
    """
    Calcula MAE, RMSE y R².
    """

    mae = mean_absolute_error(
        y_real,
        y_estimado,
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_real,
            y_estimado,
        )
    )

    r2 = r2_score(
        y_real,
        y_estimado,
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    }

modelo_base = Pipeline([
    (
        "imputacion",
        SimpleImputer(strategy="median"),
    ),
    (
        "modelo",
        DummyRegressor(strategy="median"),
    ),
])

modelo_base.fit(
    X_train,
    y_train,
)

pred_base = modelo_base.predict(X_test)

metricas_base = calcular_metricas(
    y_test,
    pred_base,
)

print("Métricas del modelo de referencia:")

for nombre, valor in metricas_base.items():
    print(f"  {nombre}: {valor:.4f}")

## 11. Random Forest

Random Forest combina múltiples árboles de decisión y puede representar:

- Relaciones no lineales.
- Interacciones entre geometría y ángulo.
- Cambios locales de presión.
- Diferencias entre fachadas.
- Efectos espaciales de las coordenadas.

Los principales hiperparámetros son:

- `n_estimators`: número de árboles.
- `max_depth`: profundidad máxima.
- `min_samples_leaf`: número mínimo de observaciones en una hoja.
- `max_features`: cantidad de variables consideradas en cada división.

Un número pequeño de observaciones por hoja puede aumentar el sobreajuste y producir superficies de predicción poco estables.

N_ESTIMATORS = 400
MAX_DEPTH = None
MIN_SAMPLES_LEAF = 2
MAX_FEATURES = 1.0

modelo_rf = Pipeline([
    (
        "imputacion",
        SimpleImputer(strategy="median"),
    ),
    (
        "modelo",
        RandomForestRegressor(
            n_estimators=N_ESTIMATORS,
            max_depth=MAX_DEPTH,
            min_samples_leaf=MIN_SAMPLES_LEAF,
            max_features=MAX_FEATURES,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    ),
])

modelo_rf.fit(
    X_train,
    y_train,
)

y_pred = modelo_rf.predict(X_test)

metricas_rf = calcular_metricas(
    y_test,
    y_pred,
)

print("Métricas de Random Forest:")

for nombre, valor in metricas_rf.items():
    print(f"  {nombre}: {valor:.4f}")


comparacion_metricas = pd.DataFrame({
    "Dummy mediana": metricas_base,
    "Random Forest": metricas_rf,
}).T

display(comparacion_metricas)

mejora_mae = (
    1
    - metricas_rf["MAE"]
    / metricas_base["MAE"]
) * 100

mejora_rmse = (
    1
    - metricas_rf["RMSE"]
    / metricas_base["RMSE"]
) * 100

print(
    f"Reducción del MAE frente a la mediana: "
    f"{mejora_mae:.2f}%"
)

print(
    f"Reducción del RMSE frente a la mediana: "
    f"{mejora_rmse:.2f}%"
)

## 13. Validación cruzada agrupada

Una única división de entrenamiento y prueba puede depender de las geometrías seleccionadas.

Se aplica `GroupKFold` para evaluar el modelo en varias combinaciones de geometrías no vistas.

Cada geometría debe aparecer en un único pliegue. Esto evita que sensores del mismo modelo físico se distribuyan entre entrenamiento y validación.

N_SPLITS = min(
    5,
    grupos_train.nunique(),
)

cv_agrupada = GroupKFold(
    n_splits=N_SPLITS
)

mae_cv = -cross_val_score(
    modelo_rf,
    X_train,
    y_train,
    groups=grupos_train,
    cv=cv_agrupada,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)

print(
    f"MAE CV por pliegue: "
    f"{np.round(mae_cv, 4)}"
)

print(
    f"MAE CV promedio: "
    f"{mae_cv.mean():.4f}"
)

print(
    f"Desviación del MAE: "
    f"{mae_cv.std():.4f}"
)

## 14. Importancia de variables por permutación

La importancia por permutación mide cuánto aumenta el error cuando se altera aleatoriamente una variable.

Se calcula utilizando el conjunto de prueba, que contiene geometrías no vistas por el modelo.

Una importancia elevada indica que el modelo depende de esa variable. Sin embargo, no demuestra por sí sola causalidad física.

Debe prestarse atención a la dependencia entre:

- `H`
- `B`
- `D`
- `H_B`

La relación `H_B` se calcula a partir de otras variables. Por tanto, la importancia puede repartirse entre variables correlacionadas.

N_REPETICIONES = 20

resultado_importancia = permutation_importance(
    modelo_rf,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=N_REPETICIONES,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

tabla_importancia = pd.DataFrame({
    "variable": VARIABLES_ENTRADA,
    "importancia_media":
        resultado_importancia.importances_mean,
    "importancia_std":
        resultado_importancia.importances_std,
})

tabla_importancia = (
    tabla_importancia
    .sort_values(
        "importancia_media",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(tabla_importancia)

N_TOP_IMPORTANCIAS = len(VARIABLES_ENTRADA)

top_importancias = (
    tabla_importancia
    .head(N_TOP_IMPORTANCIAS)
    .sort_values(
        "importancia_media",
        ascending=True,
    )
)

fig, ax = plt.subplots(figsize=(9, 6))

ax.barh(
    top_importancias["variable"],
    top_importancias["importancia_media"],
    xerr=top_importancias["importancia_std"],
    color="#2ecc71",
    alpha=0.85,
)

ax.set_xlabel(
    "Aumento del MAE al permutar la variable"
)

ax.set_ylabel("Variable")

ax.set_title(
    f"Importancia por permutación para {TARGET}"
)

plt.tight_layout()
plt.show()

print(
    "Orden de importancia:",
    tabla_importancia["variable"].tolist(),
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
)

axes[0].scatter(
    y_test,
    y_pred,
    alpha=0.55,
    s=20,
    color="#3498db",
    edgecolors="white",
    linewidth=0.25,
)

limite_inferior = min(
    y_test.min(),
    y_pred.min(),
)

limite_superior = max(
    y_test.max(),
    y_pred.max(),
)

axes[0].plot(
    [limite_inferior, limite_superior],
    [limite_inferior, limite_superior],
    "r--",
    label="Predicción perfecta",
)

axes[0].set_xlabel(f"{TARGET} observado")
axes[0].set_ylabel(f"{TARGET} predicho")

axes[0].set_title(
    f"Observado frente a predicho\n"
    f"R² = {metricas_rf['R2']:.3f}"
)

axes[0].legend()

residuos = y_test.to_numpy() - y_pred

axes[1].scatter(
    y_pred,
    residuos,
    alpha=0.55,
    s=20,
    color="#e67e22",
    edgecolors="white",
    linewidth=0.25,
)

axes[1].axhline(
    0,
    color="red",
    linestyle="--",
)

axes[1].set_xlabel(f"{TARGET} predicho")
axes[1].set_ylabel("Residuo: observado - predicho")
axes[1].set_title("Distribución de residuos")

plt.tight_layout()
plt.show()

errores = pd.DataFrame({
    "observado": y_test.to_numpy(),
    "predicho": y_pred,
    "residuo": residuos,
    "error_absoluto": np.abs(residuos),
    "geometry_id": grupos_test.to_numpy(),
})

display(
    errores
    .sort_values(
        "error_absoluto",
        ascending=False,
    )
    .head(10)
)

error_geometria = (
    errores
    .groupby("geometry_id")
    .agg(
        n=("error_absoluto", "size"),
        mae=("error_absoluto", "mean"),
        error_maximo=(
            "error_absoluto",
            "max",
        ),
    )
    .sort_values(
        "mae",
        ascending=False,
    )
)

display(error_geometria)

## 16. Modelos separados para Cp medio, RMS, mínimo y máximo

Se entrena un modelo independiente para cada objetivo.

Esto permite comparar si las mismas variables controlan:

- La presión media.
- La fluctuación.
- La succión extrema.
- La sobrepresión extrema.

Los modelos utilizan la misma división agrupada para que su comparación sea consistente.

def entrenar_objetivo(
    datos,
    target,
    features,
    group_column,
):
    """
    Entrena y evalúa un Random Forest para un objetivo.
    """

    columnas = (
        features
        + [target, group_column]
    )

    datos_modelo = (
        datos[columnas]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna(
            subset=[target, group_column]
        )
        .copy()
    )

    X_local = datos_modelo[features]
    y_local = datos_modelo[target]

    grupos_local = (
        datos_modelo[group_column]
        .astype(str)
    )

    division = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=RANDOM_STATE,
    )

    train_idx, test_idx = next(
        division.split(
            X_local,
            y_local,
            groups=grupos_local,
        )
    )

    X_train_local = X_local.iloc[train_idx]
    X_test_local = X_local.iloc[test_idx]

    y_train_local = y_local.iloc[train_idx]
    y_test_local = y_local.iloc[test_idx]

    modelo = Pipeline([
        (
            "imputacion",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "modelo",
            RandomForestRegressor(
                n_estimators=400,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ])

    modelo.fit(
        X_train_local,
        y_train_local,
    )

    prediccion = modelo.predict(
        X_test_local
    )

    metricas = calcular_metricas(
        y_test_local,
        prediccion,
    )

    importancia = permutation_importance(
        modelo,
        X_test_local,
        y_test_local,
        scoring="neg_mean_absolute_error",
        n_repeats=15,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    tabla_imp = pd.DataFrame({
        "variable": features,
        "importancia_media":
            importancia.importances_mean,
        "importancia_std":
            importancia.importances_std,
    }).sort_values(
        "importancia_media",
        ascending=False,
    )

    resultados_prediccion = pd.DataFrame({
        "observado": y_test_local.to_numpy(),
        "predicho": prediccion,
        "geometry_id":
            grupos_local.iloc[test_idx].to_numpy(),
    })

    return {
        "modelo": modelo,
        "metricas": metricas,
        "importancia": tabla_imp,
        "predicciones": resultados_prediccion,
    }

resultados = {}

for objetivo in VARIABLES_OBJETIVO:

    print(
        f"Entrenando modelo para {objetivo}..."
    )

    resultados[objetivo] = entrenar_objetivo(
        datos=df,
        target=objetivo,
        features=VARIABLES_ENTRADA,
        group_column=COLUMNA_GRUPO,
    )

print("✅ Entrenamiento completado.")

tabla_metricas = pd.DataFrame({
    objetivo: resultados[objetivo]["metricas"]
    for objetivo in VARIABLES_OBJETIVO
}).T

display(tabla_metricas.round(4))

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5),
)

tabla_metricas[["MAE", "RMSE"]].plot(
    kind="bar",
    ax=axes[0],
    color=["#3498db", "#e67e22"],
)

axes[0].set_title(
    "Comparación de errores por objetivo"
)

axes[0].set_xlabel("Variable objetivo")
axes[0].set_ylabel("Error")
axes[0].tick_params(
    axis="x",
    rotation=30,
)

tabla_metricas["R2"].plot(
    kind="bar",
    ax=axes[1],
    color="#2ecc71",
)

axes[1].axhline(
    0,
    color="black",
    linewidth=0.8,
)

axes[1].set_title(
    "Coeficiente R² por objetivo"
)

axes[1].set_xlabel("Variable objetivo")
axes[1].set_ylabel("R²")
axes[1].tick_params(
    axis="x",
    rotation=30,
)

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(
    2,
    2,
    figsize=(14, 10),
)

axes = axes.ravel()

for ax, objetivo in zip(
    axes,
    VARIABLES_OBJETIVO,
):

    importancia_objetivo = (
        resultados[objetivo]["importancia"]
        .sort_values(
            "importancia_media",
            ascending=True,
        )
    )

    ax.barh(
        importancia_objetivo["variable"],
        importancia_objetivo[
            "importancia_media"
        ],
        xerr=importancia_objetivo[
            "importancia_std"
        ],
        color=colores[objetivo],
        alpha=0.80,
    )

    ax.set_title(
        f"Importancia para {objetivo}"
    )

    ax.set_xlabel(
        "Aumento del MAE"
    )

plt.tight_layout()
plt.show()

## 17. Estabilidad de las predicciones

En el marco PCS/VDS, el modelo debe analizarse no solo por su precisión, sino también por su estabilidad.

Se estudian pequeñas perturbaciones en:

- Dirección del viento: ±1° y ±5°.
- Coordenadas: ±0.5 % y ±1 % del rango observado.

Una respuesta muy sensible puede representar:

1. Un cambio físico real cerca de una zona de separación.
2. Sobreajuste del modelo.
3. Escasez de datos en esa región.
4. Extrapolación fuera del dominio experimental.

def evaluar_perturbacion(
    modelo,
    X_original,
    variable,
    cambios,
):
    """
    Evalúa el cambio de predicción producido por
    perturbaciones en una variable.
    """

    X_base = X_original.copy()

    prediccion_base = modelo.predict(
        X_base
    )

    resultados_locales = []

    if variable == "theta":
        escala = 1.0
    else:
        escala = (
            X_base[variable].max()
            - X_base[variable].min()
        )

    for cambio in cambios:

        X_perturbado = X_base.copy()

        X_perturbado[variable] = (
            X_perturbado[variable]
            + cambio * escala
        )

        prediccion_perturbada = (
            modelo.predict(X_perturbado)
        )

        diferencia = np.abs(
            prediccion_perturbada
            - prediccion_base
        )

        resultados_locales.append({
            "variable": variable,
            "perturbacion": cambio,
            "cambio_medio":
                diferencia.mean(),
            "cambio_p95":
                np.quantile(
                    diferencia,
                    0.95,
                ),
            "cambio_maximo":
                diferencia.max(),
        })

    return pd.DataFrame(
        resultados_locales
    )

PERTURBACIONES_ANGULO = [
    -5,
    -1,
    1,
    5,
]

estabilidad_theta = evaluar_perturbacion(
    modelo=modelo_rf,
    X_original=X_test,
    variable="theta",
    cambios=PERTURBACIONES_ANGULO,
)

display(estabilidad_theta)

PERTURBACIONES_ESPACIALES = [
    -0.01,
    -0.005,
    0.005,
    0.01,
]

resultados_espaciales = []

for coordenada in ["X", "Y", "Z"]:

    resultado = evaluar_perturbacion(
        modelo=modelo_rf,
        X_original=X_test,
        variable=coordenada,
        cambios=PERTURBACIONES_ESPACIALES,
    )

    resultados_espaciales.append(
        resultado
    )

estabilidad_espacial = pd.concat(
    resultados_espaciales,
    ignore_index=True,
)

display(estabilidad_espacial)

estabilidad_total = pd.concat(
    [
        estabilidad_theta,
        estabilidad_espacial,
    ],
    ignore_index=True,
)

estabilidad_total["caso"] = (
    estabilidad_total["variable"]
    + " "
    + estabilidad_total[
        "perturbacion"
    ].astype(str)
)

fig, ax = plt.subplots(
    figsize=(12, 6)
)

sns.barplot(
    data=estabilidad_total,
    x="caso",
    y="cambio_p95",
    color="#9b59b6",
    ax=ax,
)

ax.set_xlabel("Perturbación")
ax.set_ylabel(
    "Percentil 95 del cambio absoluto en Cp"
)

ax.set_title(
    f"Estabilidad de las predicciones de {TARGET}"
)

ax.tick_params(
    axis="x",
    rotation=45,
)

plt.tight_layout()
plt.show()

RUTA_RESULTADOS = Path(
    "reports/resultados_notebook"
)

RUTA_RESULTADOS.mkdir(
    parents=True,
    exist_ok=True,
)

tabla_metricas.to_csv(
    RUTA_RESULTADOS
    / "metricas_objetivos.csv"
)

tabla_importancia.to_csv(
    RUTA_RESULTADOS
    / f"importancia_{TARGET}.csv",
    index=False,
)

errores.to_csv(
    RUTA_RESULTADOS
    / f"predicciones_{TARGET}.csv",
    index=False,
)

estabilidad_total.to_csv(
    RUTA_RESULTADOS
    / f"estabilidad_{TARGET}.csv",
    index=False,
)

print(
    f"✅ Resultados guardados en "
    f"{RUTA_RESULTADOS}"
)

mejor_variable = tabla_importancia.iloc[0][
    "variable"
]

peor_geometria = (
    error_geometria.index[0]
    if len(error_geometria) > 0
    else "No disponible"
)

print("RESUMEN DEL MODELO")
print("=" * 60)

print(f"Variable objetivo: {TARGET}")

print(
    f"MAE de prueba: "
    f"{metricas_rf['MAE']:.4f}"
)

print(
    f"RMSE de prueba: "
    f"{metricas_rf['RMSE']:.4f}"
)

print(
    f"R² de prueba: "
    f"{metricas_rf['R2']:.4f}"
)

print(
    f"Variable más importante: "
    f"{mejor_variable}"
)

print(
    f"Geometría con mayor MAE: "
    f"{peor_geometria}"
)

print(
    f"Reducción de MAE frente al modelo base: "
    f"{mejora_mae:.2f}%"
)

## Reflexión: preguntas para el análisis

1. ¿Qué ocurriría si el modelo se utiliza para una geometría cuya relación `H/B` se encuentra fuera del intervalo utilizado durante el entrenamiento?

2. ¿Puede considerarse confiable una predicción en una coordenada donde no existía una toma de presión en el ensayo original?

3. ¿Cómo debería diseñarse una validación espacial que reserve franjas completas de sensores en lugar de sensores seleccionados aleatoriamente?

4. Si `theta` resulta ser la variable más importante, ¿significa que la geometría no tiene influencia física o que su efecto se encuentra distribuido entre `H`, `B`, `D` y `H/B`?

5. ¿Por qué una correlación pequeña entre `theta` y Cp no implica necesariamente que el ángulo sea poco importante?

6. Si Random Forest presenta buen desempeño para `cp_mean`, pero un desempeño menor para `cp_min`, ¿qué fenómeno físico o estadístico podría explicar la diferencia?

7. ¿El modelo reproduce adecuadamente las succiones extremas o tiende a suavizarlas hacia la media?

8. ¿Cómo cambiarían los resultados si se reservaran direcciones completas del viento en lugar de geometrías completas?

9. ¿Qué riesgo existe al interpretar `cp_min` como un valor extremo universal si los ensayos tienen una duración limitada?

10. ¿Cómo podrían incorporarse las relaciones de vecindad entre sensores mediante una Graph Neural Network?

11. ¿Qué condiciones deberían cumplirse antes de utilizar un Fourier Neural Operator para predecir campos continuos de presión?

12. Antes de utilizar una predicción en el diseño de paneles, vidrios o anclajes, ¿cómo debería contrastarse con ensayos experimentales y normas de diseño por viento?

## Referencias

Bre, F., Gimenez, J. M., y Fachinotti, V. D. (2018). Prediction of wind pressure coefficients on building surfaces using artificial neural networks. *Energy and Buildings, 158*, 1429-1441.

Li, Z., Kovachki, N., Azizzadenesheli, K., Liu, B., Bhattacharya, K., Stuart, A., y Anandkumar, A. (2021). Fourier Neural Operator for Parametric Partial Differential Equations. *International Conference on Learning Representations, ICLR 2021*. arXiv:2010.08895.

Tokyo Polytechnic University. TPU Aerodynamic Database. Se debe citar el año de consulta y la publicación específica correspondiente al subconjunto experimental utilizado.

La referencia sobre Graph Neural Networks propuesta inicialmente debe verificarse mediante DOI, volumen, páginas o registro editorial antes de incorporarla como referencia definitiva.